In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
#test data size:
test_df = pd.read_csv("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv")
print("Number of test mashups:", len(test_df))
test_df.head()


In [ ]:
import os
import random
import librosa
import numpy as np
import librosa
import pandas as pd
import matplotlib.pyplot as plt

BASE_PATH = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"

STEMS_PATH = os.path.join(BASE_PATH, "genres_stems")
MASHUPS_PATH = os.path.join(BASE_PATH, "mashups")
TEST_CSV = os.path.join(BASE_PATH, "test.csv")
ESC_AUDIO = os.path.join(BASE_PATH, "ESC-50-master/audio")

GENRES = [
    "blues", "classical", "country", "disco", "hiphop",
    "jazz", "metal", "pop", "reggae", "rock"
]


In [ ]:
import wandb

In [ ]:

wandb.init(
    entity="ramakrishna777108-indian-institute-of-technology-madras",
    project="milestone2",
    name="mfcc-classical-ml",
    config={
        "features": "MFCC",
        "models": ["LogisticRegression", "NaiveBayes"],
        "metric": "MacroF1"
    }
)

In [ ]:
!wandb login

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

## EDA

In [ ]:
#Class distribution (songs per genre)
genre_counts = {}

for g in GENRES:
    genre_dir = os.path.join(STEMS_PATH, g)
    songs = [
        d for d in os.listdir(genre_dir)
        if os.path.isdir(os.path.join(genre_dir, d))
    ]
    genre_counts[g] = len(songs)

df_genres = pd.DataFrame.from_dict(
    genre_counts, orient="index", columns=["num_songs"]
)

df_genres


In [ ]:
#Stem availability / gaps
stem_files = ["drums.wav", "bass.wav", "vocals.wav", "others.wav"]
stem_stats = []

for g in GENRES:
    genre_dir = os.path.join(STEMS_PATH, g)
    songs = [
        d for d in os.listdir(genre_dir)
        if os.path.isdir(os.path.join(genre_dir, d))
    ]

    for s in songs:
        song_dir = os.path.join(genre_dir, s)
        available = sum(
            os.path.exists(os.path.join(song_dir, stem))
            for stem in stem_files
        )
        stem_stats.append({
            "genre": g,
            "song": s,
            "num_stems": available
        })

df_stems = pd.DataFrame(stem_stats)
df_stems["num_stems"].value_counts().sort_index()



In [ ]:
#Audio length analysis — training stems
durations = []

for g in GENRES:
    genre_dir = os.path.join(STEMS_PATH, g)
    songs = [
        d for d in os.listdir(genre_dir)
        if os.path.isdir(os.path.join(genre_dir, d))
    ]

    for s in random.sample(songs, 5):  # sample few per genre
        song_dir = os.path.join(genre_dir, s)
        for stem in stem_files:
            stem_path = os.path.join(song_dir, stem)
            if os.path.exists(stem_path):
                audio, sr = librosa.load(stem_path, sr=None)
                durations.append(len(audio) / sr)

pd.Series(durations).describe()


In [ ]:
#Mashup duration stats
test_lengths = []

for fname in test_df["filename"].sample(20, random_state=42):
    path = os.path.join(BASE_PATH, fname)
    audio, sr = librosa.load(path, sr=None)
    test_lengths.append(len(audio) / sr)

pd.Series(test_lengths).describe()


In [ ]:
#esc 50
noise_file = random.choice(os.listdir(ESC_AUDIO))
noise_path = os.path.join(ESC_AUDIO, noise_file)

noise, sr = librosa.load(noise_path, sr=22050)
print("Noise file:", noise_file)
print("Duration:", len(noise)/sr)

noise_spec = librosa.feature.melspectrogram(y=noise, sr=sr)
noise_log = librosa.power_to_db(noise_spec)

plt.figure(figsize=(10, 3))
plt.imshow(noise_log, aspect="auto", origin="lower")
plt.title("ESC-50 Noise Spectrogram")
plt.colorbar()
plt.show()


## model building

In [ ]:
import os
import random
import librosa
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report


from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB


# ================= PATHS ================= #

import os
import pandas as pd

BASE_DIR = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
STEMS_DIR = os.path.join(BASE_DIR, "genres_stems")
TEST_DIR  = os.path.join(BASE_DIR, "mashups")
TEST_CSV  = os.path.join(BASE_DIR, "test.csv")

OUT_PATH = "/kaggle/working/submission.csv"


# ================= CONFIG ================= #

SR = 22050
DURATION = 5
SAMPLES = SR * DURATION
MIN_STEMS_REQUIRED = 2

GENRES = [
    "blues", "classical", "country", "disco", "hiphop",
    "jazz", "metal", "pop", "reggae", "rock"
]

random.seed(42)
np.random.seed(42)

# ================= AUDIO UTILS ================= #

def load_audio(path):
    audio, _ = librosa.load(path, sr=SR, mono=True)
    if len(audio) < SAMPLES:
        audio = np.pad(audio, (0, SAMPLES - len(audio)))
    return audio[:SAMPLES]

def extract_mfcc(audio):
    mfcc = librosa.feature.mfcc(y=audio, sr=SR, n_mfcc=20)
    return np.mean(mfcc, axis=1)

# ================= BUILD TRAIN DATA ================= #

X_train, y_train = [], []

print("Building training data...")

for genre in GENRES:
    genre_path = os.path.join(STEMS_DIR, genre)

    song_dirs = [
        d for d in os.listdir(genre_path)
        if os.path.isdir(os.path.join(genre_path, d))
    ]

    for _ in range(100):
        chosen = random.sample(song_dirs, 4)
        mix = np.zeros(SAMPLES)
        used = 0

        for stem_name, song in zip(
            ["drums.wav", "bass.wav", "vocals.wav", "others.wav"],
            chosen
        ):
            stem_path = os.path.join(genre_path, song, stem_name)
            if not os.path.exists(stem_path):
                continue

            audio = load_audio(stem_path)
            mix += audio
            used += 1

        if used < MIN_STEMS_REQUIRED:
            continue

        mix = mix / (np.max(np.abs(mix)) + 1e-6)

        X_train.append(extract_mfcc(mix))
        y_train.append(genre)

X_train = np.array(X_train)
y_train = np.array(y_train)

print("Training samples:", X_train.shape[0])
print("Genres present:", set(y_train))

# ================= MODEL ================= #

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

# Logistic Regression
lr_model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=2000))
])

lr_model.fit(X_tr, y_tr)
lr_preds = lr_model.predict(X_val)
lr_f1 = f1_score(y_val, lr_preds, average="macro")

wandb.log({"LogisticRegression_MacroF1": lr_f1})
print("Logistic Regression Macro F1:", lr_f1)

# Naive Bayes
nb_model = GaussianNB()
nb_model.fit(X_tr, y_tr)

nb_preds = nb_model.predict(X_val)
nb_f1 = f1_score(y_val, nb_preds, average="macro")

wandb.log({"NaiveBayes_MacroF1": nb_f1})
print("Naive Bayes Macro F1:", nb_f1)


# ================= TEST INFERENCE ================= #

print("Running inference on test mashups...")
test_df = pd.read_csv(TEST_CSV)

predictions = []

for fname in test_df["filename"]:
    audio_path = os.path.join(BASE_DIR, fname)
    audio = load_audio(audio_path)
    feats = extract_mfcc(audio).reshape(1, -1)
    predictions.append(model.predict(feats)[0])

# ================= SUBMISSION ================= #

submission = pd.DataFrame({
    "id": test_df["id"],
    "genre": predictions
})

submission.to_csv("submission.csv", index=False)
print("\nSaved submission:")
wandb.finish()

In [ ]:
wandb.finish()

#milestone 2 code

In [ ]:


import os
import numpy as np
import librosa

GENRE_DIR = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"
NOISE_DIR = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio"
MASHUP_DIR = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/mashups"

# ---------- Safe audio loader ----------
def load_audio_safe(path):
    try:
        y, sr = librosa.load(path, sr=None)
        if y.size == 0 or sr is None:
            return None, None
        return y, sr
    except:
        return None, None


# ---------- 1. Mean duration of Jazz ----------
jazz_durations = []
jazz_path = os.path.join(GENRE_DIR, "jazz")

if os.path.isdir(jazz_path):
    for f in os.listdir(jazz_path):
        if f.endswith(".wav"):
            y, sr = load_audio_safe(os.path.join(jazz_path, f))
            if y is not None:
                jazz_durations.append(len(y) / sr)

mean_jazz_duration = np.mean(jazz_durations) if jazz_durations else "N/A"


# ---------- 2. Unique sample rates ----------
sample_rates = set()
for base in [GENRE_DIR, NOISE_DIR, MASHUP_DIR]:
    if not os.path.isdir(base):
        continue
    for root, _, files in os.walk(base):
        for f in files:
            if f.endswith(".wav"):
                _, sr = load_audio_safe(os.path.join(root, f))
                if sr is not None:
                    sample_rates.add(sr)

sample_rates = sorted(sample_rates) if sample_rates else "N/A"


# ---------- 3. Zero-byte audio files ----------
zero_byte_files = 0
for root, _, files in os.walk(GENRE_DIR):
    for f in files:
        path = os.path.join(root, f)
        if f.endswith(".wav") and os.path.getsize(path) == 0:
            zero_byte_files += 1


# ---------- 4. Vocal peak amplitude ----------
# Dataset DOES NOT contain vocal stems
avg_vocal_peak_db = "N/A (no vocal stems provided)"


# ---------- 5 & 6. Spectral centroid ----------
genre_centroids = {}

for genre in os.listdir(GENRE_DIR):
    gpath = os.path.join(GENRE_DIR, genre)
    if not os.path.isdir(gpath):
        continue

    values = []
    for f in os.listdir(gpath):
        if f.endswith(".wav"):
            y, sr = load_audio_safe(os.path.join(gpath, f))
            if y is not None:
                sc = librosa.feature.spectral_centroid(y=y, sr=sr)
                if sc.size > 0:
                    values.append(np.mean(sc))

    if values:
        genre_centroids[genre] = np.mean(values)

mean_blues_centroid = genre_centroids.get("blues", "N/A")
highest_centroid_genre = max(genre_centroids, key=genre_centroids.get) if genre_centroids else "N/A"


# ---------- 7. Silence in first 0.5 seconds ----------
silence_files = 0
for root, _, files in os.walk(GENRE_DIR):
    for f in files:
        if f.endswith(".wav"):
            y, sr = load_audio_safe(os.path.join(root, f))
            if y is not None:
                first_half_sec = y[:int(0.5 * sr)]
                if first_half_sec.size > 0 and np.max(np.abs(first_half_sec)) < 1e-4:
                    silence_files += 1


# ---------- PRINT ANSWERS ----------
print("Mean Jazz duration (sec):", mean_jazz_duration)
print("Unique sample rates:", sample_rates)
print("Zero-byte files in train:", zero_byte_files)
print("Average vocal peak amplitude (dB):", avg_vocal_peak_db)
print("Mean spectral centroid (Blues):", mean_blues_centroid)
print("Genre with highest spectral centroid:", highest_centroid_genre)
print("Files silent in first 0.5s:", silence_files)